In [10]:
import polars as pl
import numpy as np

pl.Config.set_tbl_rows(-1)
pl.Config.set_float_precision(2)
pl.Config.set_decimal_separator(',')
pl.Config.set_thousands_separator('.')

ENDERECO_DADOS = './../../dados/'
ENDERECO_VOTACAO = './../../dados/'

In [ ]:


# Obtendo os dados
try:
    print("Obtendo os dados...")
    df_bolsa_familia = pl.scan_parquet(ENDERECO_DADOS + 'bolsa_familia.parquet')
    # print(df_bolsa_familia)

    df_dados_votacao = pl.read_csv(
        ENDERECO_VOTACAO + 'votacao_secao_2022_BR.csv', 
        separator=';', encoding='iso-8859-1'
    )
    print(df_dados_votacao.columns)

    # Filtrar p/ segundo turno 'NR_TURNO' e nº do candidato 'NR_VOTAVEL' 13 e 22
    df_votacao_turno2 = df_dados_votacao.filter(
        (pl.col("NR_TURNO") == 2) &
        (pl.col("NR_VOTAVEL").is_in([13, 22]))
    )
    # print(df_votacao_turno2)

except Exception as e:
    print(f"Erro ao obter os daos: {e}")

#### Processando Votação


In [ ]:
# Processando os dados de votação
try:
    # Delimitar as variáveis, converter Categorical, agrupar/totalizar

    # Delimitando as variáveis SG_UF, NM_VOTAVEL, QT_VOTOS
    df_votacao = df_votacao_turno2.lazy().select(
        ['SG_UF', 'NM_VOTAVEL', 'QT_VOTOS']
    )

    # print(df_votacao)

    # Converte para Categorical
    df_votacao = df_votacao.with_columns(
        pl.col("SG_UF").cast(pl.Categorical),
        pl.col("NM_VOTAVEL").cast(pl.Categorical)
    )

    # Agrupar e totalizar os votos por estado e candidato
    df_votacao = (
        df_votacao.group_by(['SG_UF', 'NM_VOTAVEL'])
        .agg(pl.sum('QT_VOTOS'))
        .sort('SG_UF', descending=False)
    )

    # Executa o plano de execução e coleta os resultados em um DataFrame
    # df_votacao = df_votacao.collect()  
    # display(df_votacao)


except Exception as e:
    print(f"Erro ao processar os dados de votação: {e}")

In [ ]:
# Processando os dados de bolsa família
try:
    # Delimitar as variáveis, converter Categorical, agrupar/totalizar

    # Delimitando as variáveis 'UF', 'VALOR PARCELA'
    df_bolsa_familia = df_bolsa_familia.lazy().select(
        ['UF', 'VALOR PARCELA']
    )

    # Converte para Categorical
    df_bolsa_familia = df_bolsa_familia.with_columns(
        pl.col("UF").cast(pl.Categorical)
    )

    # Agrupar e totalizar os valores por estado
    df_bolsa_familia = (
        df_bolsa_familia.group_by('UF')
        .agg(pl.sum('VALOR PARCELA'))
        .sort('UF', descending=False)
    )

    # Executa o plano de execução e coleta os resultados em um DataFrame
    # df_bolsa_familia = df_bolsa_familia.collect()  
    # display(df_bolsa_familia)

except Exception as e:
    print(f"Erro ao processar os dados de bolsa família: {e}")

In [ ]:
# Unir os DataFrames de votação e bolsa família
try:
    # Unir votação e bolsa família
    df_votos_bolsa_familia = df_votacao.join(df_bolsa_familia, left_on='SG_UF', right_on='UF')

    # coleta os dados
    df_votos_bolsa_familia = (
        df_votos_bolsa_familia.collect()
        .sort('VALOR PARCELA', descending=True)
    )
    display(df_votos_bolsa_familia)
except Exception as e:
    print(f"Erro ao unir os DataFrames: {e}")

# Calculando a Correlação

In [ ]:
# Calculando a correlação:
try:
    dict_correlacao = {}
    
    for candidato in df_votos_bolsa_familia["NM_VOTAVEL"].unique():
        # Filtrar os dados para o candidato atual
        df_candidato = (
            df_votos_bolsa_familia.filter(pl.col("NM_VOTAVEL") == candidato)
        )
        
        # Criar o array de valores de bolsa familia e votos
        array_votos = np.array(df_candidato["QT_VOTOS"])
        array_bolsa_familia = np.array(df_candidato["VALOR PARCELA"])
        
        # Calcular a correlação de Pearson
        # O resultado é uma matriz de correlação, então pegamos o valor da primeira linha, na segunda coluna [0,1] para obter a correlação
        correlacao = np.corrcoef(array_votos, array_bolsa_familia)[0, 1]
        
        print(f"Correlação do candidato {candidato}: {correlacao:.4f}")
        
        dict_correlacao[candidato] = correlacao
        
    
except Exception as e:
    print(f"Erro ao calcular a correlação: {e}")